# UltraGliClass stage 3: long context (8,192 tokens) + many options (<= 256), from the stage-2 FULL arm C model

**Inputs:** code dataset `ultragliclass-code` **v8** (has `configs/stage3_longctx.yaml`) + the output of kernel `ultragliclass-s2full`
**version 4** (has `runs/stage2_full_C/final_model` AND `data/stage2_full/train.jsonl` + `label_pools.json`, used for replay).
**MODE = "smoke"** (current account, ~1.5 GPU-h, hard cap 1.8 h + upload): REAL data + REAL config, S2 baseline on the long-Jev slice, ~1 h of
training (a few hundred of the 956 steps; s/step, peak memory, NaN), checkpoint + `needs_resume`, long-Jev eval of that checkpoint.
**MODE = "full"** is DEFERRED to Kaggle account #2: attach this smoke output, `PREV_OUTPUT="auto"` resumes it. See STAGE3_JOB_SPEC.md.
**Partly in-domain** (inherited from stage 2 through replay): typed-decisions customer_service + agent_trace_observability; massive and financial_phrasebank are NOT zero-shot.


In [ ]:
# ---- knobs ----
MODE          = "full"       # "smoke" (current account, ~1.5 GPU-h) or "full" (account #2, resumes the smoke output); STAGE3_JOB_SPEC.md
ARMS          = ["s3"]       # ONE arm: configs/stage3_longctx.yaml (do not change)
S3_CONFIG     = "configs/stage3_longctx.yaml"
SMOKE_TRAIN_HOURS = 1.0      # smoke: train cap (train.py checkpoints + exits needs_resume); ~250 steps at ~14 s/step (EST)
SMOKE_SESSION_HOURS = 1.8    # smoke: whole-session cap (setup + data + baseline + train + checkpoint long-Jev eval); + upload <= 2 h
SMOKE_QUICK_EVAL = False     # smoke: also run the 0.12-h suite quick eval on the checkpoint (adds ~0.12 h)
INIT_MODEL    = "auto"     # S2 final_model dir, OR "auto" = search /kaggle/input for runs/stage2_full_C/final_model (kernel ultragliclass-s2full v4)
DATA_RECIPE   = "configs/data_stage3.yaml"
DATA_DIR      = "data/stage3"
LONGJEV_SLICE = "calib/jev_long_test.jsonl"   # 431 rows, built on the box 25 Oct (scripts/build_long_jev_slice.py); md5 checked below
LONGJEV_MD5   = "cc664b35683a6703d4a8be15983fd5cb"
LONGJEV_MAXLEN = 8192
SEMANTIC_DATA_APPROVED = True   # replay carries S2's LocalLLaMA/typed-decisions TRAIN rows (approved 24 Oct 2026, labelled in-domain)
OOM_RETRY_SMALLER_BATCH = True  # on OOM: ONE resume with data.max_length=4096 (batch is already 1); a 2nd OOM stops the session
PREP_ARGS     = ""                              # smoke tests: "--streaming"
CALIB_FILE    = "calib/stage1_val.jsonl"       # = data/stage1/val.jsonl used by the S1 and ModernBERT evals (same global-T fit)
CODE_INPUT    = "auto"     # auto-detect the code dataset under /kaggle/input
PREV_OUTPUT   = "auto"     # "auto" = previous version's output of THIS kernel (runs/stage3_*/STATUS.json or results/stage3_*), else fresh
USE_SINGLE_GPU = False     # True = never launch 2 processes (python -u train.py on GPU 0, grad-accum doubled)
AUTO_FALLBACK_SINGLE_GPU = True   # 2-GPU run stalls/crashes (not OOM) => resume on 1 GPU in the same session
DIST_CHECK    = True       # 20-s NCCL pre-flight before the first 2-GPU launch
NCCL_P2P_DISABLE = False   # True = start with NCCL P2P off (auto-retried if the pre-flight hangs)
STALL_MINUTES = 15         # watchdog: no output for this long => kill the run
SESSION_HOURS = 12.0
RESERVE_HOURS = 0.5
ARM_MAX_HOURS = 10.6       # train budget per SESSION (train.py saves a checkpoint and exits needs_resume; next session resumes)
TOTAL_MAX_HOURS = 10.8     # STOP: session clock cap incl. setup, data, train, eval; + ~0.2 h output upload => <= 11 h (GPU-hours = session hours)
EVAL_RESERVE_HOURS = 0.2   # kept free for the quick eval after an arm
QUICK_EVAL    = True
EVAL_MAX_PER_DATASET = 500
EVAL_PERMS    = 3
EVAL_CALIB_N  = 2000             # rows of CALIB_FILE used to fit the global temperature (S1: 2000)
EVAL_TYPED_MAX_CASES = None      # None = all 400 cases / 2,000 decisions (as in S1)
DROP_CKPT_AFTER_ARM = True       # delete checkpoint-* only once the run is COMPLETE and its final eval is written (final_model is kept)
MID_EVAL      = True             # quick eval of the newest checkpoint at the end of an unfinished session
EXTRA_SET     = []         # dotted overrides for every arm, e.g. ["train.max_steps=20"]
ARM_SET       = {}         # per-arm overrides, e.g. {"C": ["data.consistency.pair_prob=0.1"]}
import os, time, glob, json, sys, shutil, subprocess
T0 = time.time()
LOG_DIR = "/kaggle/working/logs"; os.makedirs(LOG_DIR, exist_ok=True)
NB_LOG = open(os.path.join(LOG_DIR, f"notebook_{time.strftime('%Y%m%d_%H%M%S')}.log"), "a", buffering=1)
def say(*a):
    line = f"[nb {time.strftime('%H:%M:%S')} +{(time.time() - T0) / 60:.1f}m] " + " ".join(str(x) for x in a)
    print(line, flush=True); NB_LOG.write(line + "\n")

def find_code_input():
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train.py", recursive=True)]
    hits = [h for h in hits if os.path.exists(os.path.join(h, "configs", "stage3_longctx.yaml"))]   # needs the v8 code (stage-3 configs)
    hits = [h for h in hits if not os.path.isdir(os.path.join(h, "runs"))] or hits
    hits.sort(key=lambda p: (0 if "ultragliclass-code" in p else 1, len(p)))
    say("code candidates:", hits)
    return hits[0] if hits else None

def find_prev_output():
    hits = sorted({(p.split("/runs/")[0] if "/runs/" in p else p.split("/results/")[0]) for p in glob.glob("/kaggle/input/**/runs/stage3_*/STATUS.json", recursive=True) + glob.glob("/kaggle/input/**/results/stage3_baseline_s2_longjev_8192.json", recursive=True)}, key=len)
    say("previous stage-3 output candidates:", hits)
    return hits[0] if hits else None

def find_init_model():
    cands = []
    if INIT_MODEL != "auto":
        cands = [INIT_MODEL] if os.path.exists(os.path.join(INIT_MODEL, "config.json")) else \
            [os.path.dirname(p) for p in glob.glob(os.path.join(INIT_MODEL, "**", "final_model", "config.json"), recursive=True)]
    else:
        cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/stage2_full_C/final_model/config.json", recursive=True)]
    cands.sort(key=len)
    say("S2 model candidates:", cands)
    return cands[0] if cands else None

if CODE_INPUT == "auto":
    CODE_INPUT = find_code_input()
if PREV_OUTPUT == "auto":
    PREV_OUTPUT = find_prev_output()
INIT_SRC = find_init_model()
assert ARMS == ["s3"] and MODE in ("smoke", "full"), (ARMS, MODE)
say("CODE_INPUT =", CODE_INPUT, "| INIT_SRC =", INIT_SRC, "| PREV_OUTPUT =", PREV_OUTPUT, "| ARMS =", ARMS, "| USE_SINGLE_GPU =", USE_SINGLE_GPU)
assert CODE_INPUT and os.path.exists(os.path.join(CODE_INPUT, "configs", "stage3_longctx.yaml")), "v8 code dataset (with configs/stage3_longctx.yaml) not found under /kaggle/input"
assert INIT_SRC, "S2 model not found: attach the output of kernel ultragliclass-s2full v4 (runs/stage2_full_C/final_model) or set INIT_MODEL"

def find_s2_data():
    c = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/data/stage2_full/train.jsonl", recursive=True)}, key=len)
    say("S2 data candidates:", c)
    return c[0] if c else None
S2_DATA = find_s2_data()
assert S2_DATA and os.path.exists(os.path.join(S2_DATA, "label_pools.json")), \
    "S2 data (data/stage2_full/train.jsonl + label_pools.json) not found: attach the output of kernel ultragliclass-s2full v4"
if MODE == "smoke":   # same data + config + schedule as the full run; only the time caps differ (resume-safe first chunk)
    ARM_MAX_HOURS = SMOKE_TRAIN_HOURS; TOTAL_MAX_HOURS = SMOKE_SESSION_HOURS; QUICK_EVAL = SMOKE_QUICK_EVAL
say("MODE", MODE, "| S2_DATA", S2_DATA, "| EXTRA_SET", EXTRA_SET)


In [ ]:
import shutil, subprocess
WORK = "/kaggle/working/ultragliclass"
if not os.path.exists(os.path.join(WORK, "train.py")):
    shutil.copytree(CODE_INPUT, WORK, dirs_exist_ok=True, copy_function=shutil.copyfile,
                    ignore=shutil.ignore_patterns("runs", "data", "smoke_logs", "results", "__pycache__"))
subprocess.run(f"chmod -R u+w {WORK}", shell=True)  # copytree copies read-only dir modes from /kaggle/input
os.chdir(WORK)
say("work dir", WORK)
torch_pin = subprocess.run("pip freeze | grep -E '^torch=='", shell=True, capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(torch_pin + "\n")
say("torch pin:", torch_pin)
subprocess.run("pip install -q -c /tmp/constraints.txt -r requirements.txt", shell=True, check=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)
import torch, transformers
NGPU = torch.cuda.device_count()
say("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", NGPU, "transformers", transformers.__version__)

In [ ]:
# The S2 model: validate it, then COPY it out of /kaggle/input (plain file copies, in this single process, before torchrun) so
# no rank ever reads the read-only input mount. INIT_LOCAL is what every arm gets as model.init_from.
INIT_LOCAL = "/tmp/ugc_init_model"
need = ["config.json", "tokenizer.json", "tokenizer_config.json"]
for f in need:
    assert os.path.exists(os.path.join(INIT_SRC, f)), f"S2 model dir {INIT_SRC} lacks {f}"
weights = [f for f in os.listdir(INIT_SRC) if f.endswith((".safetensors", ".bin"))]
assert weights, f"no weight file in {INIT_SRC}: {os.listdir(INIT_SRC)}"
size = sum(os.path.getsize(os.path.join(INIT_SRC, f)) for f in os.listdir(INIT_SRC))
assert size > 200e6, f"S2 model dir is only {size / 1e6:.0f} MB: incomplete"
cfg_s2 = json.load(open(os.path.join(INIT_SRC, "config.json")))
say("S2 model:", INIT_SRC, f"{size / 2**20:.0f} MB", weights, "| architectures", cfg_s2.get("architectures"), "| max_num_classes", cfg_s2.get("max_num_classes"))
shutil.rmtree(INIT_LOCAL, ignore_errors=True); os.makedirs(INIT_LOCAL)
for f in os.listdir(INIT_SRC):
    s = os.path.join(INIT_SRC, f)
    if os.path.isfile(s):
        shutil.copyfile(s, os.path.join(INIT_LOCAL, f))
        assert os.path.getsize(os.path.join(INIT_LOCAL, f)) == os.path.getsize(s), f"copy size mismatch {f}"
subprocess.run(f"chmod -R u+w {INIT_LOCAL}", shell=True)
say("S2 model copied to", INIT_LOCAL, sorted(os.listdir(INIT_LOCAL)))

In [ ]:
# Restore the previous output. Runs HERE, in the single notebook process, BEFORE torchrun: no rank ever touches
# /kaggle/input. Plain file copies (no read-only mode bits from the input mount), then chmod u+w, then verify sizes.
def restore(src_root):
    for sub in ("runs", "data", "results"):
        src = os.path.join(src_root, sub)
        if not os.path.isdir(src):
            say("restore: no", sub, "in", src_root); continue
        n = b = 0; t = time.time()
        for dirpath, _, files in os.walk(src):
            rel = os.path.relpath(dirpath, src_root)
            os.makedirs(rel, exist_ok=True)
            for f in files:
                s, d = os.path.join(dirpath, f), os.path.join(rel, f)
                size = os.path.getsize(s)
                if not (os.path.exists(d) and os.path.getsize(d) == size):
                    shutil.copyfile(s, d)
                assert os.path.getsize(d) == size, f"size mismatch {d}"
                n += 1; b += size
                if size > 50e6: say(f"restore: copied {d} ({size / 2**20:.0f} MB)")
        say(f"restore: {sub}: {n} files, {b / 2**30:.2f} GB in {time.time() - t:.0f}s")
    subprocess.run("chmod -R u+w runs data results 2>/dev/null; true", shell=True)

if PREV_OUTPUT:
    say("restore from", PREV_OUTPUT)
    restore(PREV_OUTPUT)
for st in glob.glob("runs/*/STATUS.json"):
    s = json.load(open(st)); say("found", st, s.get("state"), "step", s.get("global_step"), "last", s.get("last_checkpoint"))
for c in sorted(glob.glob("runs/*/checkpoint-*")):
    say("checkpoint", c, sorted(os.listdir(c)))

In [ ]:
# Data. Stage-2 mix is linked read-only for replay; the long-Jev eval slice ships in the code dataset (md5-checked).
import hashlib
sys.path.insert(0, WORK)
from ugc.config import load_config as _lc
os.makedirs("results", exist_ok=True); os.makedirs("data", exist_ok=True)
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
    return h.hexdigest()
assert SEMANTIC_DATA_APPROVED
assert os.path.exists("calib/stage2_pilot_val.jsonl") and os.path.exists(LONGJEV_SLICE), "calib files missing from the code dataset"
assert md5(LONGJEV_SLICE) == LONGJEV_MD5, f"{LONGJEV_SLICE} md5 {md5(LONGJEV_SLICE)} != {LONGJEV_MD5}"
if not os.path.exists("data/stage2_full"):
    os.symlink(S2_DATA, "data/stage2_full")           # read-only is fine: prepare_data only reads (replay + pools copy)
say("data/stage2_full ->", os.path.realpath("data/stage2_full"), sorted(os.listdir("data/stage2_full")))
recipe, ddir = DATA_RECIPE, DATA_DIR   # smoke and full share the REAL recipe
MD5_REC = f"results/{os.path.basename(ddir)}_data_md5.json"
if not os.path.exists(os.path.join(ddir, "train.jsonl")):
    say("preparing data:", recipe); t = time.time()
    subprocess.run(f"python -u prepare_data.py --recipe {recipe}", shell=True, check=True)
    say(f"data prep took {(time.time() - t) / 60:.1f} min")
DATA_MD5 = md5(os.path.join(ddir, "train.jsonl"))
if os.path.exists(MD5_REC):
    assert json.load(open(MD5_REC))["train_md5"] == DATA_MD5, "STOP: rebuilt stage-3 data differs from session 1 (resume order would differ)"
else:
    json.dump({"train_md5": DATA_MD5, "recipe": recipe}, open(MD5_REC, "w"))
man = json.load(open(os.path.join(ddir, "manifest.json")))
shutil.copyfile(os.path.join(ddir, "manifest.json"), f"results/{os.path.basename(ddir)}_data_manifest.json")
used = {k: v for k, v in man["sources"].items() if v.get("train_rows", 0) > 0}
bad = [k for k in used if any(x in k.lower() for x in ("dbpedia", "bitext", "banking77", "clinc", "massive", "snips", "phrasebank", "sst5", "procedural-typed-decisions"))]
slice_txt = {json.loads(l)["text"] for l in open(LONGJEV_SLICE)}
val_txt = {json.loads(l)["text"] for l in open(os.path.join(ddir, "val.jsonl"))}
n_rows = n_long = n_slice = n_val = n_sa = 0
with open(os.path.join(ddir, "train.jsonl")) as f:
    for line in f:
        ex = json.loads(line); n_rows += 1
        n_long += ex["source"].startswith("tasksource/tasksource-jev-typed-decisions")
        n_slice += ex["text"] in slice_txt; n_val += ex["text"] in val_txt
cfg_t = _lc(S3_CONFIG, [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET)["train"]
eff = int(cfg_t["per_device_train_batch_size"]) * int(cfg_t["gradient_accumulation_steps"]) * (NGPU if NGPU >= 2 and not USE_SINGLE_GPU else 1)
STEPS_PLANNED = int(cfg_t.get("max_steps") or 0) or (n_rows // eff)
say(f"train rows {n_rows} (long Jev {n_long}, replay {n_rows - n_long}) | val {man['val_rows']} | pools {man.get('taxonomy')} | labels/row {man['labels_per_row']}")
say(f"leak: train rows whose text is in the long-Jev EVAL slice {n_slice}, in the stage-3 val {n_val} | steps planned {STEPS_PLANNED} x {eff}")
say("sources used:", {k: v["train_rows"] for k, v in used.items()})
json.dump({"mode": MODE, "train_rows": n_rows, "long_jev_rows": n_long, "replay_rows": n_rows - n_long, "train_in_eval_slice": n_slice,
           "train_in_val": n_val, "md5": DATA_MD5, "steps_planned": STEPS_PLANNED, "licenses": {k: v.get("license") for k, v in used.items()}},
          open(f"results/{os.path.basename(ddir)}_data_check.json", "w"), indent=1)
assert not bad, f"excluded / eval-set / dedicated procedural source in the data: {bad}"
assert n_slice == 0 and n_val == 0, "STOP: eval-slice or val text found in train"
if True:
    assert 20_000 <= n_long <= 26_000 and 39_000 <= n_rows - n_long <= 40_000, (n_long, n_rows - n_long)


In [ ]:
# Launch with live, unbuffered streaming to this notebook AND to /kaggle/working/logs/, a no-output watchdog, a NCCL pre-flight,
# a single-GPU fallback, then the quick eval, one arm after the other.
import signal, threading, queue, re
sys.path.insert(0, WORK)
from ugc.config import load_config
LAST_OOM = False

def run_streamed(cmd, env_extra, log_name, stall_min):
    global LAST_OOM
    LAST_OOM = False
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
           "TORCH_NCCL_DUMP_ON_TIMEOUT": "1", "NCCL_DEBUG": "WARN", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", **env_extra}
    path = os.path.join(LOG_DIR, f"{log_name}_{time.strftime('%H%M%S')}.log")
    say("RUN", cmd, "| env", env_extra, "| log", path)
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, start_new_session=True)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in p.stdout], q.put(None)), daemon=True).start()
    last, stalled = time.time(), False
    with open(path, "a", buffering=1) as f:
        while True:
            try:
                line = q.get(timeout=10)
            except queue.Empty:
                if time.time() - last > stall_min * 60:
                    say(f"WATCHDOG: no output for {stall_min} min -> killing process group"); stalled = True
                    os.killpg(p.pid, signal.SIGTERM); time.sleep(30)
                    if p.poll() is None: os.killpg(p.pid, signal.SIGKILL)
                    break
                continue
            if line is None: break
            if re.search(r"OutOfMemoryError|CUDA out of memory", line): LAST_OOM = True
            last = time.time(); print(line, end="", flush=True); f.write(line)
    rc = p.wait()
    say("EXIT", rc, "stalled" if stalled else "", "OOM" if LAST_OOM else "")
    return rc, stalled

def elapsed_h():
    return (time.time() - T0) / 3600

def arm_budget_h():
    return min(ARM_MAX_HOURS, TOTAL_MAX_HOURS - elapsed_h() - EVAL_RESERVE_HOURS, SESSION_HOURS - elapsed_h() - RESERVE_HOURS)

def arm_sets(arm):
    return [f"model.init_from={INIT_LOCAL}"] + EXTRA_SET + ARM_SET.get(arm, [])

SMALL_BATCH = False
def train_cmd(arm, single):
    sets = [f"train.time_budget_hours={max(0.1, arm_budget_h()):.2f}"] + arm_sets(arm)
    if SMALL_BATCH:   # OOM retry: batch is already 1 -> shorten the context (fallback; reported as a deviation)
        sets += ["data.max_length=4096"]
    accum = int(load_config(S3_CONFIG, arm_sets(arm))["train"].get("gradient_accumulation_steps", 1))
    if single and NGPU > 1:
        sets = [s for s in sets if not s.startswith("train.gradient_accumulation_steps=")] + [f"train.gradient_accumulation_steps={accum * NGPU}"]
    launcher = "python -u" if single else f"torchrun --standalone --nproc_per_node {NGPU}"
    return f"{launcher} train.py --config {S3_CONFIG} --resume auto --set {' '.join(sets)}"

single = USE_SINGLE_GPU or NGPU < 2
nccl_env = {"NCCL_P2P_DISABLE": "1"} if NCCL_P2P_DISABLE else {}
if not single and DIST_CHECK:
    rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check", 3)
    if rc != 0 and not nccl_env:
        say("pre-flight failed: retrying with NCCL_P2P_DISABLE=1"); nccl_env = {"NCCL_P2P_DISABLE": "1"}
        rc, st = run_streamed(f"torchrun --standalone --nproc_per_node {NGPU} scripts/dist_check.py", nccl_env, "dist_check_p2poff", 3)
    if rc != 0:
        say("pre-flight failed twice: using 1 GPU"); single = True

def read_status(out_dir):
    p = os.path.join(out_dir, "STATUS.json")
    return json.load(open(p)) if os.path.exists(p) else {}

def quick_eval(model_dir, res_path, tag):
    cmd = (f"python -u eval.py --model {model_dir} --suites gliclass intent typed --max_per_dataset {EVAL_MAX_PER_DATASET} "
           f"--permutations {EVAL_PERMS} --calib_file {CALIB_FILE} --calib_n {EVAL_CALIB_N} --max_length 1024 --out {res_path}" +
           (f" --typed_max_cases {EVAL_TYPED_MAX_CASES}" if EVAL_TYPED_MAX_CASES else ""))
    rc, _ = run_streamed(cmd, {}, f"quick_eval_{tag}", 30)
    if not os.path.exists(res_path):
        return {"eval_rc": rc}
    r = json.load(open(res_path)); d = r["datasets"]; t1 = (r.get("typed_decisions") or {}).get("T1", {})
    return {"eval_rc": rc, "avg_macro_f1": r.get("gliclass_avg_macro_f1"), "global_T": r.get("global_T"), "typed_acc": t1.get("accuracy"),
            "typed_by_type": t1.get("accuracy_by_type"), "typed_by_workflow": t1.get("accuracy_by_workflow"),
            "flip_many_label": {k: d[k]["perm_flip_rate"] for k in ("banking77", "clinc150", "massive", "SetFit/20_newsgroups") if k in d},
            "NOT_zero_shot": ["massive (HWU64 shares 56/59 intents)", "financial_phrasebank (fin-news sentiment, same 3 labels)",
                              "typed-decisions customer_service + agent_trace_observability (trained on their TRAIN split)"]}

def longjev_eval(model_dir, res_path, tag):
    cmd = f"python -u scripts/eval_long_jev.py --model {model_dir} --slice {LONGJEV_SLICE} --max_length {LONGJEV_MAXLEN} --batch_size 1 --dtype fp16 --out {res_path}"
    rc, _ = run_streamed(cmd, {"CUDA_VISIBLE_DEVICES": "0"}, f"longjev_{tag}", 20)
    if not os.path.exists(res_path):
        return {"rc": rc}
    x = json.load(open(res_path))
    return {"rc": rc, "acc": x["accuracy"], "ci95": x["accuracy_ci95_cluster"], "n": x["n"], "truncated_frac": x["truncated_frac"], "seconds": x["seconds"]}

BASE_LJ = "results/stage3_baseline_s2_longjev_8192.json"
def baseline():
    if os.path.exists(BASE_LJ):
        say("baseline long-Jev eval of S2 already present:", BASE_LJ); return {"note": "present"}
    say("BASELINE: stage-2 final_model on the long-Jev slice at", LONGJEV_MAXLEN)
    return longjev_eval(INIT_LOCAL, BASE_LJ, "baseline_s2")

def smoke_report(out_dir):
    rows = []
    for p in sorted(glob.glob(os.path.join(LOG_DIR, "train_s3_*.log"))):
        for line in open(p, errors="ignore"):
            if "sec_per_step" in line or "peak_reserved_gb" in line: rows.append(line.strip()[:300])
    say("speed/memory metric lines (last 12):"); [print(x) for x in rows[-12:]]
    st = read_status(out_dir); hb = os.path.join(out_dir, "HEARTBEAT.json")
    return {"status": st.get("state"), "step": st.get("global_step"), "heartbeat": json.load(open(hb)) if os.path.exists(hb) else None,
            "metric_lines": rows[-12:], "oom": bool(LAST_OOM), "small_ctx_retry": SMALL_BATCH}

def run_arm(arm):
    global single, SMALL_BATCH
    cfg = load_config(S3_CONFIG, arm_sets(arm))
    out_dir = cfg["train"]["output_dir"]
    st0 = read_status(out_dir)
    rec = {"arm": arm, "mode": MODE, "out_dir": out_dir, "state": None, "t0_hours": round(elapsed_h(), 3), "start_step": st0.get("global_step", 0)}
    rec["baseline_longjev"] = baseline()
    st = st0
    if st.get("state") == "complete" and os.path.isdir(st.get("final_model", "")):
        say("STAGE 3 already complete (restored): skipping training")
    else:
        if arm_budget_h() < 0.3:
            rec["state"] = "not_started_budget"; say(f"NOT STARTED, only {arm_budget_h():.2f} h of budget left"); return rec
        say(f"===== STAGE 3 {MODE}: step {rec['start_step']} -> {STEPS_PLANNED}, session train budget {arm_budget_h():.2f} h =====")
        rc, stalled = run_streamed(train_cmd(arm, single), ({"CUDA_VISIBLE_DEVICES": "0"} if single else nccl_env),
                                   f"train_{arm}_{'1gpu' if single else 'ddp'}", STALL_MINUTES)
        st = read_status(out_dir)
        if LAST_OOM and OOM_RETRY_SMALLER_BATCH and not SMALL_BATCH and st.get("state") != "complete" and arm_budget_h() >= 0.3:
            say("OOM at 8192 -> ONE resume from the newest checkpoint with data.max_length=4096 (FALLBACK, report it)")
            SMALL_BATCH = True; rec["oom_retry_maxlen4096"] = True
            rc, stalled = run_streamed(train_cmd(arm, single), ({"CUDA_VISIBLE_DEVICES": "0"} if single else nccl_env),
                                       f"train_{arm}_ctx4096", STALL_MINUTES)
            st = read_status(out_dir)
        if (rc != 0 or stalled) and not LAST_OOM and not single and AUTO_FALLBACK_SINGLE_GPU and st.get("state") not in ("complete", "nan_stop") \
                and arm_budget_h() >= 0.4 and MODE == "full":
            say("2-GPU run failed/stalled -> resuming on 1 GPU from the newest checkpoint (2x slower: report before relying on it)")
            single = True
            rc, stalled = run_streamed(train_cmd(arm, True), {"CUDA_VISIBLE_DEVICES": "0"}, f"train_{arm}_1gpu_fallback", STALL_MINUTES)
            st = read_status(out_dir)
        rec["oom"] = bool(LAST_OOM)
    rec["state"] = st.get("state", "no_status"); rec["status"] = st; rec["end_step"] = st.get("global_step")
    print(json.dumps(st, indent=2))
    rec["speed_mem"] = smoke_report(out_dir)   # sec_per_step / peak_reserved_gb lines, heartbeat, OOM / 4096-fallback flags
    if rec["state"] == "complete":
        res_path = "results/stage3_final_quick.json"
        if QUICK_EVAL and not os.path.exists(res_path):
            rec["eval"] = quick_eval(st["final_model"], res_path, "final")
        lj = "results/stage3_final_longjev_8192.json"
        if not os.path.exists(lj):
            rec["longjev"] = longjev_eval(st["final_model"], lj, "final")
        if os.path.exists(lj) and os.path.exists(BASE_LJ):
            subprocess.run(f"python scripts/eval_long_jev.py --compare {BASE_LJ} {lj} > results/stage3_longjev_compare.json", shell=True)
            rec["longjev_compare"] = json.load(open("results/stage3_longjev_compare.json"))
        if DROP_CKPT_AFTER_ARM and os.path.exists(res_path) and os.path.exists(lj):
            for c in glob.glob(os.path.join(out_dir, "checkpoint-*")):
                shutil.rmtree(c, ignore_errors=True)
    else:
        say(f"STAGE 3: state={rec['state']} at step {rec['end_step']} (needs_resume -> attach this output, run the next session; nan_stop/no_status -> report)")
        ck = sorted(glob.glob(os.path.join(out_dir, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
        if MID_EVAL and ck and rec["state"] == "needs_resume" and SESSION_HOURS - elapsed_h() > 0.3:
            step = int(ck[-1].rsplit("-", 1)[1])
            rec["longjev_mid"] = longjev_eval(ck[-1], f"results/stage3_mid_step{step}_longjev_8192.json", f"mid{step}")
            if os.path.exists(BASE_LJ) and os.path.exists(f"results/stage3_mid_step{step}_longjev_8192.json"):
                _o = f"results/stage3_mid_step{step}_longjev_compare.json"
                subprocess.run(f"python scripts/eval_long_jev.py --compare {BASE_LJ} results/stage3_mid_step{step}_longjev_8192.json > {_o}", shell=True)
                rec["longjev_mid_compare"] = json.load(open(_o))
            if QUICK_EVAL and SESSION_HOURS - elapsed_h() > 0.3:
                rec["eval_mid"] = quick_eval(ck[-1], f"results/stage3_mid_step{step}_quick.json", f"mid{step}")
    rec["wall_hours"] = round(elapsed_h() - rec["t0_hours"], 3)
    return rec

SESSION_IDX = 1 + len(glob.glob(f"results/stage3_{MODE}_summary_s*.json"))
SUMMARY = []
def write_summary():
    json.dump({"mode": MODE, "session": SESSION_IDX, "session_hours": round(elapsed_h(), 3), "arms": SUMMARY, "init_model_src": INIT_SRC,
               "prev_output": PREV_OUTPUT, "s2_data": S2_DATA, "data_md5": DATA_MD5, "partly_in_domain": True},
              open(f"results/stage3_{MODE}_summary_s{SESSION_IDX}.json", "w"), indent=2)

say(f"STAGE 3 {MODE} SESSION {SESSION_IDX}")
rec = run_arm("s3"); SUMMARY.append(rec); write_summary()
say(f"SESSION done: mode={MODE} state={rec['state']} steps {rec.get('start_step')} -> {rec.get('end_step')} wall={rec.get('wall_hours')} h")

In [ ]:
print(json.dumps({"mode": MODE, "session": SESSION_IDX, "session_hours": round(elapsed_h(), 2),
                  "arms": [{k: v for k, v in r.items() if k != "status"} for r in SUMMARY]}, indent=2, default=str))
for r in SUMMARY:
    s = r.get("status", {})
    say(f"STAGE 3 {MODE}: {r['state']} step {s.get('global_step')}/{s.get('max_steps')} train_loss {s.get('train_loss')} "
        f"eval_loss {(s.get('final_eval') or {}).get('eval_loss')} | baseline {r.get('baseline_longjev')} | longjev {r.get('longjev')} | compare {r.get('longjev_compare')}")

In [ ]:
if os.path.islink("data/stage2_full"):
    os.unlink("data/stage2_full")   # never save the 1.8 GB S2 data link into this output
for out in sorted(glob.glob("runs/stage3_*")):          # unfinished arms keep their newest checkpoint only
    ck = sorted(glob.glob(os.path.join(out, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
    for c in ck[:-1]:
        shutil.rmtree(c)
subprocess.run("du -sh runs data results /kaggle/working/logs 2>/dev/null", shell=True)
say("session hours used:", round(elapsed_h(), 2)); NB_LOG.close()